# AMASS Preprocessed Data Visualization

This notebook reads and visualizes preprocessed AMASS joint data using matplotlib for 3D visualization and animation.

## Features
- Load preprocessed .npz joint data files
- 3D joint position visualization
- Skeleton connection display
- Motion sequence animation
- Statistical data analysis


In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.animation as animation
import json
import glob
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Configure matplotlib parameters
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("Libraries imported successfully!")


In [ ]:
# Configuration: data paths and parameters
DATA_ROOT = "../datasets/AMASS_preproc"  # Preprocessed data root directory
NUM_JOINTS = 22  # Number of joints

# Visualization parameters
JOINT_SIZE = 50   # Joint marker size
LINE_WIDTH = 2    # Skeleton line width
COLORS = {
    'joints': 'red',
    'bones': 'blue',
    'pelvis': 'green'
}

# Verify data path exists
data_path = Path(DATA_ROOT)
if data_path.exists():
    print(f"Data path exists: {DATA_ROOT}")
    # Count npz files
    npz_files = list(data_path.rglob("*_joints22.npz"))
    print(f"Found {len(npz_files)} preprocessed data files")
else:
    print(f"Data path not found: {DATA_ROOT}")
    print("Please ensure the preprocessing script has been run!")
    npz_files = []


In [ ]:
# Load preprocessed joint data
def load_joints_data(npz_path: str) -> Tuple[Optional[np.ndarray], Dict]:
    """
    Load preprocessed joint data file.
    
    Args:
        npz_path: Path to .npz file.
        
    Returns:
        joints: Joint position data with shape (T, num_joints, 3)
        meta: Metadata dictionary
    """
    try:
        data = np.load(npz_path)
        joints = data['joints']
        
        # Parse metadata
        meta_str = data['meta'].item() if 'meta' in data else "{}"
        meta = json.loads(meta_str)
        
        # Convert fp16 to float32 if needed
        if joints.dtype == np.float16:
            joints = joints.astype(np.float32)
            
        print(f"Loaded: {Path(npz_path).name}")
        print(f"   - Shape: {joints.shape}")
        print(f"   - Dtype: {joints.dtype}")
        print(f"   - Frames: {meta.get('num_frames', 'unknown')}")
        print(f"   - Stride: {meta.get('frame_stride', 'unknown')}")
        
        return joints, meta
        
    except Exception as e:
        print(f"Failed to load: {npz_path}")
        print(f"   Error: {e}")
        return None, {}


# Test loading a sample file
if npz_files:
    sample_file = npz_files[min(10, len(npz_files) - 1)]
    sample_joints, sample_meta = load_joints_data(str(sample_file))
else:
    print("No data files found")
    sample_joints, sample_meta = None, {}


In [ ]:
# Display metadata information
def display_metadata(meta: Dict) -> None:
    """Display metadata information."""
    print("Metadata Information:")
    print("=" * 40)
    for key, value in meta.items():
        if key == 'source_file':
            print(f"  Source file: {Path(value).name}")
        else:
            print(f"  {key}: {value}")
    print("=" * 40)


# Display sample metadata
if sample_meta:
    display_metadata(sample_meta)


In [ ]:
# 3D Joint Visualization Functions

# SMPL-X first 22 joint connections (skeleton)
JOINT_CONNECTIONS = [
    # Spine connections
    (0, 3),   # pelvis -> spine1
    (3, 6),   # spine1 -> spine2
    (6, 9),   # spine2 -> spine3
    (9, 12),  # spine3 -> neck
    (12, 15), # neck -> head
    
    # Left arm
    (9, 13),  # spine3 -> left_collar
    (13, 16), # left_collar -> left_shoulder
    (16, 18), # left_shoulder -> left_elbow
    (18, 20), # left_elbow -> left_wrist
    
    # Right arm
    (9, 14),  # spine3 -> right_collar
    (14, 17), # right_collar -> right_shoulder
    (17, 19), # right_shoulder -> right_elbow
    (19, 21), # right_elbow -> right_wrist
    
    # Left leg
    (0, 1),   # pelvis -> left_hip
    (1, 4),   # left_hip -> left_knee
    (4, 7),   # left_knee -> left_ankle
    (7, 10),  # left_ankle -> left_foot
    
    # Right leg
    (0, 2),   # pelvis -> right_hip
    (2, 5),   # right_hip -> right_knee
    (5, 8),   # right_knee -> right_ankle
    (8, 11),  # right_ankle -> right_foot
]


def plot_3d_pose(joints: np.ndarray, frame_idx: int = 0, title: str = "3D Human Pose") -> None:
    """
    Plot 3D human pose.
    
    Args:
        joints: Joint data with shape (T, num_joints, 3)
        frame_idx: Frame index to display
        title: Plot title
    """
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # Get joint positions for specified frame
    pose = joints[frame_idx]  # (num_joints, 3)
    
    # Plot joint points
    x, y, z = pose[:, 0], pose[:, 1], pose[:, 2]
    ax.scatter(x, y, z, c=COLORS['joints'], s=JOINT_SIZE, alpha=0.8)
    
    # Draw skeleton connections
    for start_idx, end_idx in JOINT_CONNECTIONS:
        if start_idx < len(pose) and end_idx < len(pose):
            ax.plot3D(
                [pose[start_idx, 0], pose[end_idx, 0]],
                [pose[start_idx, 1], pose[end_idx, 1]], 
                [pose[start_idx, 2], pose[end_idx, 2]],
                color=COLORS['bones'], linewidth=LINE_WIDTH
            )
    
    # Highlight pelvis joint
    ax.scatter(pose[0, 0], pose[0, 1], pose[0, 2], 
              c=COLORS['pelvis'], s=JOINT_SIZE*2, alpha=1.0, marker='s')
    
    # Set axis labels
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.set_zlabel('Z (m)')
    ax.set_title(f"{title} - Frame {frame_idx}")
    
    # Set equal aspect ratio
    max_range = np.array([x.max()-x.min(), y.max()-y.min(), z.max()-z.min()]).max() / 2.0
    mid_x, mid_y, mid_z = (x.max()+x.min())*0.5, (y.max()+y.min())*0.5, (z.max()+z.min())*0.5
    ax.set_xlim(mid_x - max_range, mid_x + max_range)
    ax.set_ylim(mid_y - max_range, mid_y + max_range)
    ax.set_zlim(mid_z - max_range, mid_z + max_range)
    
    plt.tight_layout()
    plt.show()


print("3D visualization functions defined!")


In [ ]:
# Visualize single frame joint positions
if sample_joints is not None:
    print("Visualizing first frame of sample data:")
    plot_3d_pose(sample_joints, frame_idx=0, title="AMASS Frame")
    
    # Also show middle frame if available
    if sample_joints.shape[0] > 1:
        mid_frame = sample_joints.shape[0] // 2
        plot_3d_pose(sample_joints, frame_idx=mid_frame, title="AMASS Frame")
else:
    print("No sample data available")


In [ ]:
# Create motion sequence animation
def create_pose_animation(joints: np.ndarray, interval: int = 100, max_frames: int = 100):
    """
    Create joint position animation.
    
    Args:
        joints: Joint data with shape (T, num_joints, 3)
        interval: Animation frame interval in milliseconds
        max_frames: Maximum number of frames to display
        
    Returns:
        Animation object
    """
    # Limit frames for performance
    num_frames = min(joints.shape[0], max_frames)
    joints_subset = joints[:num_frames]
    
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # Calculate data range for axis limits
    all_points = joints_subset.reshape(-1, 3)
    x_min, x_max = all_points[:, 0].min(), all_points[:, 0].max()
    y_min, y_max = all_points[:, 1].min(), all_points[:, 1].max()
    z_min, z_max = all_points[:, 2].min(), all_points[:, 2].max()
    
    # Set axis range
    ax.set_xlim(x_min - 0.1, x_max + 0.1)
    ax.set_ylim(y_min - 0.1, y_max + 0.1)
    ax.set_zlim(z_min - 0.1, z_max + 0.1)
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.set_zlabel('Z (m)')
    
    def animate(frame):
        ax.clear()
        
        # Reset axis limits after clear
        ax.set_xlim(x_min - 0.1, x_max + 0.1)
        ax.set_ylim(y_min - 0.1, y_max + 0.1)
        ax.set_zlim(z_min - 0.1, z_max + 0.1)
        ax.set_xlabel('X (m)')
        ax.set_ylabel('Y (m)')
        ax.set_zlabel('Z (m)')
        
        pose = joints_subset[frame]
        x, y, z = pose[:, 0], pose[:, 1], pose[:, 2]
        
        # Plot joint points
        ax.scatter(x, y, z, c=COLORS['joints'], s=JOINT_SIZE, alpha=0.8)
        
        # Draw skeleton connections
        for start_idx, end_idx in JOINT_CONNECTIONS:
            if start_idx < len(pose) and end_idx < len(pose):
                ax.plot3D(
                    [pose[start_idx, 0], pose[end_idx, 0]],
                    [pose[start_idx, 1], pose[end_idx, 1]], 
                    [pose[start_idx, 2], pose[end_idx, 2]],
                    color=COLORS['bones'], linewidth=LINE_WIDTH
                )
        
        # Highlight pelvis
        ax.scatter(pose[0, 0], pose[0, 1], pose[0, 2], 
                  c=COLORS['pelvis'], s=JOINT_SIZE*2, alpha=1.0, marker='s')
        
        ax.set_title(f'AMASS Motion Sequence - Frame {frame}/{num_frames-1}')
    
    # Create animation
    anim = animation.FuncAnimation(fig, animate, frames=num_frames, 
                                  interval=interval, blit=False, repeat=True)
    
    plt.tight_layout()
    plt.show()
    
    return anim


# Create animation for sample data
if sample_joints is not None and sample_joints.shape[0] > 1:
    print("Creating motion sequence animation...")
    print(f"   Total frames: {sample_joints.shape[0]}")
    
    # Create animation (limit to 50 frames for performance)
    animation_obj = create_pose_animation(sample_joints, interval=200, max_frames=50)
else:
    print("Not enough frame data to create animation")


In [ ]:
# Batch process multiple sequences
def batch_visualize_sequences(data_root: str, max_sequences: int = 5) -> None:
    """
    Batch load and visualize multiple sequences.
    
    Args:
        data_root: Data root directory
        max_sequences: Maximum number of sequences to process
    """
    npz_files = list(Path(data_root).rglob("*_joints22.npz"))
    
    if not npz_files:
        print("No data files found")
        return
    
    print(f"Found {len(npz_files)} data files")
    
    # Limit processing count
    selected_files = npz_files[:max_sequences]
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12), subplot_kw={'projection': '3d'})
    axes = axes.flatten()
    
    for i, npz_file in enumerate(selected_files):
        if i >= 6:  # Maximum 6 subplots
            break
            
        joints, meta = load_joints_data(str(npz_file))
        
        if joints is not None:
            ax = axes[i]
            
            # Select middle frame for display
            mid_frame = joints.shape[0] // 2
            pose = joints[mid_frame]
            
            # Draw joints and skeleton
            x, y, z = pose[:, 0], pose[:, 1], pose[:, 2]
            ax.scatter(x, y, z, c=COLORS['joints'], s=30, alpha=0.8)
            
            for start_idx, end_idx in JOINT_CONNECTIONS:
                if start_idx < len(pose) and end_idx < len(pose):
                    ax.plot3D(
                        [pose[start_idx, 0], pose[end_idx, 0]],
                        [pose[start_idx, 1], pose[end_idx, 1]], 
                        [pose[start_idx, 2], pose[end_idx, 2]],
                        color=COLORS['bones'], linewidth=1
                    )
            
            # Set axis labels
            ax.set_xlabel('X')
            ax.set_ylabel('Y')
            ax.set_zlabel('Z')
            ax.set_title(f"{Path(npz_file).stem[:20]}...\n{joints.shape[0]} frames", fontsize=8)
            
            # Set equal aspect ratio
            max_range = np.array([x.max()-x.min(), y.max()-y.min(), z.max()-z.min()]).max() / 2.0
            mid_x, mid_y, mid_z = (x.max()+x.min())*0.5, (y.max()+y.min())*0.5, (z.max()+z.min())*0.5
            ax.set_xlim(mid_x - max_range, mid_x + max_range)
            ax.set_ylim(mid_y - max_range, mid_y + max_range)
            ax.set_zlim(mid_z - max_range, mid_z + max_range)
    
    # Hide unused subplots
    for j in range(len(selected_files), len(axes)):
        axes[j].set_visible(False)
    
    plt.tight_layout()
    plt.suptitle('AMASS Data Sequence Preview', fontsize=16, y=0.98)
    plt.show()


# Run batch visualization
if npz_files:
    print("Batch visualizing sequences...")
    batch_visualize_sequences(DATA_ROOT, max_sequences=6)
else:
    print("No data available for batch visualization")


In [ ]:
# Statistical analysis and data exploration
def analyze_motion_statistics(joints: np.ndarray, meta: Dict) -> None:
    """
    Analyze joint motion statistical properties.
    
    Args:
        joints: Joint data with shape (T, num_joints, 3)
        meta: Metadata dictionary
    """
    print("Motion Statistical Analysis")
    print("=" * 50)
    
    # Basic statistics
    T, J, _ = joints.shape
    print(f"Frames: {T}")
    print(f"Joints: {J}")
    print(f"Frame stride: {meta.get('frame_stride', 'unknown')}")
    
    # Calculate motion range
    joint_ranges = np.ptp(joints, axis=0)  # (num_joints, 3)
    max_range_per_joint = np.max(joint_ranges, axis=1)
    
    print(f"\nJoint Motion Range Statistics:")
    print(f"  Max range: {max_range_per_joint.max():.3f}m (joint {max_range_per_joint.argmax()})")
    print(f"  Min range: {max_range_per_joint.min():.3f}m (joint {max_range_per_joint.argmin()})")
    print(f"  Mean range: {max_range_per_joint.mean():.3f}m")
    
    # Calculate velocity (frame difference)
    if T > 1:
        velocities = np.diff(joints, axis=0)  # (T-1, num_joints, 3)
        speeds = np.linalg.norm(velocities, axis=2)  # (T-1, num_joints)
        
        print(f"\nMotion Speed Statistics:")
        print(f"  Max speed: {speeds.max():.3f}m/frame")
        print(f"  Mean speed: {speeds.mean():.3f}m/frame")
        print(f"  Speed std: {speeds.std():.3f}m/frame")
    
    # Visualize statistical results
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # 1. Joint motion range bar chart
    ax1 = axes[0, 0]
    joint_names = [f"J{i}" for i in range(J)]
    ax1.bar(joint_names, max_range_per_joint)
    ax1.set_title('Max Motion Range per Joint')
    ax1.set_ylabel('Range (m)')
    ax1.tick_params(axis='x', rotation=45)
    
    # 2. Pelvis trajectory (joint 0)
    ax2 = axes[0, 1]
    pelvis_traj = joints[:, 0, :]  # (T, 3)
    ax2.plot(pelvis_traj[:, 0], pelvis_traj[:, 1], 'b-', alpha=0.7, label='XY trajectory')
    ax2.scatter(pelvis_traj[0, 0], pelvis_traj[0, 1], c='green', s=100, label='Start')
    ax2.scatter(pelvis_traj[-1, 0], pelvis_traj[-1, 1], c='red', s=100, label='End')
    ax2.set_title('Pelvis Motion Trajectory (XY plane)')
    ax2.set_xlabel('X (m)')
    ax2.set_ylabel('Y (m)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. Speed distribution histogram
    if T > 1:
        ax3 = axes[1, 0]
        ax3.hist(speeds.flatten(), bins=50, alpha=0.7, edgecolor='black')
        ax3.set_title('Joint Speed Distribution')
        ax3.set_xlabel('Speed (m/frame)')
        ax3.set_ylabel('Frequency')
        ax3.grid(True, alpha=0.3)
    
    # 4. Joint height variation (Z coordinate)
    ax4 = axes[1, 1]
    heights = joints[:, :, 2]  # (T, num_joints)
    for j in [0, 1, 2, 15]:  # Show height changes for key joints
        if j < J:
            ax4.plot(heights[:, j], label=f'Joint {j}', alpha=0.8)
    ax4.set_title('Key Joint Height Variation')
    ax4.set_xlabel('Frame')
    ax4.set_ylabel('Height (m)')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


# Perform statistical analysis on sample data
if sample_joints is not None:
    print("Starting statistical analysis...")
    analyze_motion_statistics(sample_joints, sample_meta)
else:
    print("No data available for statistical analysis")


## Usage Guide and Summary

### Main Features
1. **Data Loading**: Read preprocessed AMASS joint data (.npz format)
2. **3D Visualization**: Single frame joint positions and skeleton connections
3. **Animation Display**: Dynamic visualization of joint motion sequences
4. **Batch Processing**: Visualize multiple motion sequences simultaneously
5. **Statistical Analysis**: Motion range, speed distribution, and other statistics

### Data Format
- Input files: `*_joints22.npz`
- Joint data: `(T, 22, 3)` - T frames, 22 joints, XYZ coordinates
- Metadata: JSON format, includes source file, frame count, stride, etc.

### Custom Usage

```python
# Load specific file
joints, meta = load_joints_data("path/to/your/file_joints22.npz")

# Visualize specific frame
plot_3d_pose(joints, frame_idx=50)

# Create animation
anim = create_pose_animation(joints, interval=100, max_frames=100)

# Statistical analysis
analyze_motion_statistics(joints, meta)
```

### Notes
- Ensure the preprocessing script has been run
- For large datasets, process in batches
- Animation generation may take some time
- Visualization parameters can be adjusted as needed
